# PD01 · Explorar y preparar AGROSAVIA

[Índice y distribución de las cuatro horas](../ejercicios/README.md) · [Guía de pandas](../docs/manual-pandas.md)

**Tiempo de trabajo: 60 minutos.** El objetivo es reconocer la estructura de un DataFrame y preparar medidas y fechas conservando los valores originales.

| Símbolo | Uso |
|:---:|---|
| 📘 | Concepto o función de apoyo. |
| ✅ | Evidencia que debe quedar visible. |
| ⚠️ | Límite de interpretación. |

Esta plantilla se encuentra en `ejercicios/PD01_exploracion_agrosavia.ipynb`. Antes de comenzar, se crea una copia en `soluciones/PD01_exploracion_agrosavia.ipynb` dentro del repositorio de cada estudiante. El desarrollo y la entrega se realizan en esa copia; la plantilla de `ejercicios/` se conserva como enunciado. En JupyterLab se puede duplicar el archivo y mover la copia a `soluciones/`, manteniendo el nombre indicado; si ya existe una solución, se continúa sobre ella. Las instrucciones se presentan en celdas Markdown; las celdas de código marcadas como **Desarrollo** contienen el espacio de trabajo. Cada actividad incluye una celda para explicar el resultado.

Se utiliza el kernel del entorno `.venv` del curso. Los CSV ya deben estar descargados mediante `kit/00_datos.py`. La celda inicial prepara las rutas y carga únicamente las entradas de este ejercicio; el desarrollo del análisis corresponde al estudiante. Este notebook se ejecuta de forma independiente.

**Apoyo en el manual:** secciones 1–5 y la receta de registros en revisión de la sección 11.

## Ubicación de trabajo: la raíz del repositorio

📘 **La raíz del repositorio** es la carpeta principal de la copia del proyecto. En la instalación del curso se llama `big-data-postgraduate` y contiene `README.md`, `kit/`, `ejercicios/` y `soluciones/`:

```text
bigdata/
└── big-data-postgraduate/       ← raíz del repositorio
    ├── README.md
    ├── kit/
    ├── ejercicios/              ← enunciados y plantillas
    └── soluciones/              ← notebooks resueltos
```

**Desde la terminal de Ubuntu en WSL.** `pwd` muestra la carpeta actual, `cd` cambia de carpeta y `ls` muestra su contenido. Los siguientes comandos se ejecutan en la terminal, no en una celda Python:

```bash
pwd
cd /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
pwd
ls
```

✅ Después de `cd`, el segundo `pwd` debe mostrar `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. En la salida de `ls` deben aparecer `README.md`, `kit`, `ejercicios` y `soluciones`. La carpeta `bigdata` es la carpeta contenedora; la raíz del proyecto es su subcarpeta `big-data-postgraduate`.

La ruta anterior corresponde a la configuración de la guía WSL. Si el repositorio se clonó en otra ubicación, se sustituye por la ruta real de esa copia. Si `cd` informa que la carpeta no existe, se revisa esa ubicación antes de continuar.

**Desde una subcarpeta conocida.** `..` significa «carpeta superior». Se comprueba primero la ubicación con `pwd` y se utiliza la alternativa correspondiente:

| Ubicación actual, relativa al repositorio | Comando para volver a la raíz |
|---|---|
| `ejercicios/`, `soluciones/` o `kit/` | `cd ..` |
| `kit/Notebooks/` | `cd ../..` |

Al llegar a la raíz, se puede activar el entorno y abrir JupyterLab desde la terminal:

```bash
source .venv/bin/activate
jupyter lab
```

**Dentro de un notebook ya abierto.** La terminal y el kernel de Python mantienen sus propios directorios de trabajo. La celda de carga siguiente localiza la raíz y la guarda en `REPO`, aunque el notebook esté abierto desde `soluciones/` o `ejercicios/`. Después de ejecutar esa celda, el siguiente bloque puede ejecutarse en una celda Python para comprobar la ubicación y cambiar el directorio del kernel a la raíz:

```python
import os
from pathlib import Path

print("Directorio antes del cambio:", Path.cwd())
print("Raíz detectada:", REPO)
os.chdir(REPO)
print("Directorio después del cambio:", Path.cwd())
```

✅ `Path.cwd()` debe coincidir con `REPO` después del cambio. Esta operación cambia la base para interpretar rutas relativas; el archivo del notebook permanece en `soluciones/`. La carga de datos de estos ejercicios también funciona sin ese cambio porque construye las rutas a partir de `REPO`.

## Preparación

Se ejecuta la siguiente celda y se comprueba que el intérprete pertenece al entorno del curso y que pandas corresponde a la versión 2.2.3. Las rutas se calculan desde el repositorio, sin introducir rutas personales de Windows, WSL o macOS.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD01"
archivos = ['agrosavia.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
suelo = pd.read_csv(RAW / "agrosavia.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python
pandas: 2.2.3
Repositorio: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD01


## 1. Inspección inicial · 10 minutos

Presentar la cantidad de filas y columnas, los nombres de las columnas, sus tipos y cinco registros. Explicar qué representa una fila y por qué la lectura inicial conserva los campos como texto.

📘 **Apoyo:** `shape`, `columns`, `dtypes`, `head` e `info`.

✅ **Comprobación:** El corte del manifiesto contiene 92.738 filas y 32 columnas. La tabla original debe conservar esa cantidad de filas durante el ejercicio.

In [2]:
# 1. Guardar y mostrar dimensiones iniciales
filas_originales, columnas_originales = suelo.shape
print(f"Dimensiones de AGROSAVIA: {filas_originales:,} filas y {columnas_originales} columnas")

# Control de conservación de filas requeridas
assert len(suelo) == 92738, f"Se esperaban 92.738 filas, pero se encontraron {len(suelo)}"

# 2. Nombres de columnas y tipos de datos iniciales
print("\n--- Nombres de columnas y tipos de datos iniciales ---")
print(suelo.dtypes)

# 3. Muestra de los primeros 5 registros
print("\n--- Primeros 5 registros ---")
display(suelo.head(5))

# 4. Resumen estructural del DataFrame
print("\n--- Información general del DataFrame ---")
suelo.info()

Dimensiones de AGROSAVIA: 92,738 filas y 32 columnas

--- Nombres de columnas y tipos de datos iniciales ---
Secuencial                            object
Fecha de Análisis                     object
Departamento                          object
Municipio                             object
Cultivo                               object
Estado                                object
Tiempo de establecimiento             object
Topografia                            object
Drenaje                               object
Riego                                 object
Fertilizantes aplicados               object
pH agua:suelo                         object
Materia organica                      object
Fósforo Bray II                       object
Azufre Fosfato monocalcico            object
Acidez Intercambiable                 object
Aluminio intercambiable               object
Calcio intercambiable                 object
Magnesio intercambiable               object
Potasio intercambiable              

,Secuencial,Fecha de Análisis,Departamento,Municipio,Cultivo,Estado,Tiempo de establecimiento,Topografia,Drenaje,Riego,...,Conductividad electrica,Hierro disponible olsen,Cobre disponible,Manganeso disponible Olsen,Zinc disponible Olsen,Boro disponible,Hierro disponible doble acido,Cobre disponible doble acido,Manganeso disponible doble acido,Zinc disponible doble acido
0,1,7/01/2014,NARIÑO,SAN ANDRÉS DE TUMACO,No Indica,No indica,No indica,No indica,No indica,No indica,...,0.133,66.39,1.700,1.700,1.5,0.187,ND,ND,ND,ND
1,2,21/08/2014,HUILA,SANTA MARÍA,Granadilla,Establecido,De 1 a 5 años,Pendiente,Buen drenaje,No Tiene,...,2.749,250.9,3.7,29.4,28.70,2.084,ND,ND,ND,ND
2,3,22/08/2014,ANTIOQUIA,LIBORINA,Café,Por establecer,No indica,Pendiente,Mal drenaje,No Tiene,...,0.328,390,11.10,8.299,5,0.085,ND,ND,ND,ND
3,4,22/08/2014,ANTIOQUIA,LIBORINA,Maracuyá,Por establecer,No indica,Ondulado,Mal drenaje,No Tiene,...,0.171,200,4.2,4.699,1,0.229,ND,ND,ND,ND
4,5,22/08/2014,ANTIOQUIA,LIBORINA,Café,Por establecer,No indica,Ondulado,Mal drenaje,No Tiene,...,0.323,117.9,3.8,4.600,0.700,0.139,ND,ND,ND,ND



--- Información general del DataFrame ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92738 entries, 0 to 92737
Data columns (total 32 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   Secuencial                          92738 non-null  object
 1   Fecha de Análisis                   92738 non-null  object
 2   Departamento                        92738 non-null  object
 3   Municipio                           92738 non-null  object
 4   Cultivo                             92738 non-null  object
 5   Estado                              92738 non-null  object
 6   Tiempo de establecimiento           92738 non-null  object
 7   Topografia                          92738 non-null  object
 8   Drenaje                             92738 non-null  object
 9   Riego                               92738 non-null  object
 10  Fertilizantes aplicados             92738 non-null  object
 11  pH agua:sue

**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

## 2. Diagnóstico de cinco campos · 10 minutos

Construir una tabla llamada `perfil` para `Secuencial`, `Fecha de Análisis`, `Departamento`, `Municipio` y `pH agua:suelo`. Para cada campo se registran los nulos de pandas, los textos vacíos después de retirar espacios y los marcadores `ND`. Las tres situaciones se cuentan por separado.

📘 **Apoyo:** `isna`, `.str.strip`, `.str.upper`, `.eq` y `sum`.

✅ **Comprobación:** `perfil` debe tener una fila por campo. Con `keep_default_na=False`, un texto vacío o `ND` no se convierte automáticamente en un nulo.

In [6]:
import pandas as pd

# Lista con los nombres exactos de los 5 campos requeridos
campos_p02 = [
    'Secuencial',
    'Fecha de Análisis',
    'Departamento',
    'Municipio',
    'pH agua:suelo'
]

registros_perfil = []

for col in campos_p02:
    serie = suelo[col]
    
    # 1. Nulos nativos de pandas
    nulos_pandas = serie.isna().sum()
    
    # 2. Limpieza de espacios alrededor del texto
    serie_limpia = serie.fillna('').astype(str).str.strip()
    
    # 3. Textos vacíos ("")
    textos_vacios = (serie_limpia == '').sum() - nulos_pandas
    if textos_vacios < 0:
        textos_vacios = (serie_limpia == '').sum()
        
    # 4. Marcadores "ND"
    marcadores_nd = (serie_limpia.str.upper() == 'ND').sum()
    
    # Total de valores incompletos
    total_incompletos = nulos_pandas + textos_vacios + marcadores_nd
    
    registros_perfil.append({
        'campo': col,
        'nulos_pandas': nulos_pandas,
        'textos_vacios': textos_vacios,
        'marcadores_nd': marcadores_nd,
        'total_incompletos': total_incompletos,
        'pct_incompletos': f"{(total_incompletos / len(suelo)) * 100:.2f}%"
    })

# Guardar y mostrar el DataFrame 'perfil'
perfil = pd.DataFrame(registros_perfil)
display(perfil)

,campo,nulos_pandas,textos_vacios,marcadores_nd,total_incompletos,pct_incompletos
0,Secuencial,0,0,0,0,0.00%
1,Fecha de Análisis,0,0,0,0,0.00%
2,Departamento,0,0,0,0,0.00%
3,Municipio,0,2,0,2,0.00%
4,pH agua:suelo,0,0,0,0,0.00%


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

## 3. Conversión de pH y fecha · 15 minutos

Crear `ph` desde `pH agua:suelo` y `fecha_dt` desde `Fecha de Análisis`. Utilizar conversión numérica con `errors="coerce"` y fecha con formato `%d/%m/%Y`. Conservar los originales. Distinguir entradas vacías de textos no vacíos que no pudieron convertirse.

📘 **Apoyo:** `pd.to_numeric`, `.str.replace`, `pd.to_datetime`, `isna` y máscaras booleanas.

✅ **Comprobación:** Se muestran los conteos de conversiones fallidas y hasta cinco ejemplos con su valor original. No se reemplazan medidas ausentes por cero.

In [7]:
import pandas as pd

# 1. Conversión de pH
# Reemplazar coma decimal por punto y aplicar pd.to_numeric con coerce
ph_limpio = suelo['pH agua:suelo'].astype(str).str.strip().str.replace(',', '.', regex=False)
suelo['ph'] = pd.to_numeric(ph_limpio, errors='coerce')

# Identificar fallas en pH
ph_estaba_vacio = suelo['pH agua:suelo'].isna() | (suelo['pH agua:suelo'].astype(str).str.strip().isin(['', 'ND', 'nd']))
ph_fallos_conversion = suelo['ph'].isna() & ~ph_estaba_vacio

print(f"--- CONVERSIÓN DE pH ---")
print(f"Total registros con pH numérico válido: {suelo['ph'].notna().sum():,}")
print(f"Total entradas vacías o ND originales: {ph_estaba_vacio.sum():,}")
print(f"Total fallos de conversión (textos no vacíos no convertidos): {ph_fallos_conversion.sum():,}")

if ph_fallos_conversion.sum() > 0:
    print("\nHasta 5 ejemplos de valores originales de pH que no pudieron convertirse:")
    display(suelo.loc[ph_fallos_conversion, ['pH agua:suelo']].head(5))

# 2. Conversión de Fecha
# Convertir con formato día/mes/año (%d/%m/%Y)
fecha_limpia = suelo['Fecha de Análisis'].astype(str).str.strip()
suelo['fecha_dt'] = pd.to_datetime(fecha_limpia, format='%d/%m/%Y', errors='coerce')

# Identificar fallas en Fecha
fecha_estaba_vacia = suelo['Fecha de Análisis'].isna() | (suelo['Fecha de Análisis'].astype(str).str.strip().isin(['', 'ND', 'nd']))
fecha_fallos_conversion = suelo['fecha_dt'].isna() & ~fecha_estaba_vacia

print("\n--- CONVERSIÓN DE FECHA DE ANÁLISIS ---")
print(f"Total registros con fecha válida: {suelo['fecha_dt'].notna().sum():,}")
print(f"Total entradas vacías o ND originales: {fecha_estaba_vacia.sum():,}")
print(f"Total fallos de conversión (fechas no válidas/corruptas): {fecha_fallos_conversion.sum():,}")

if fecha_fallos_conversion.sum() > 0:
    print("\nHasta 5 ejemplos de valores originales de Fecha que no pudieron convertirse:")
    display(suelo.loc[fecha_fallos_conversion, ['Fecha de Análisis']].head(5))

--- CONVERSIÓN DE pH ---
Total registros con pH numérico válido: 92,727
Total entradas vacías o ND originales: 0
Total fallos de conversión (textos no vacíos no convertidos): 11

Hasta 5 ejemplos de valores originales de pH que no pudieron convertirse:


,pH agua:suelo
7282,4..01
46528,>10.00
46530,>10.00
52903,<3.80
56330,<3.80



--- CONVERSIÓN DE FECHA DE ANÁLISIS ---
Total registros con fecha válida: 92,738
Total entradas vacías o ND originales: 0
Total fallos de conversión (fechas no válidas/corruptas): 0


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

## 4. Registros para revisión · 15 minutos

Crear la columna booleana `en_revision`. Se marca una fila si su pH es ausente, está fuera de `[0, 14]` o su fecha no se pudo interpretar. Contar por separado cada motivo y el total de filas marcadas. Conservar todas las observaciones.

📘 **Apoyo:** `between`, `isna`, `|`, `~`, `.loc` y `value_counts`.

✅ **Comprobación:** Las filas marcadas y no marcadas deben sumar el total original. Una misma fila puede tener varios motivos; por eso la suma de los motivos no necesariamente coincide con el total marcado.

In [8]:
import pandas as pd

# 1. Definición de las condiciones individuales de revisión
cond_ph_ausente = suelo['ph'].isna()
cond_ph_fuera_rango = ~suelo['ph'].isna() & ~suelo['ph'].between(0, 14)
cond_fecha_invalida = suelo['fecha_dt'].isna()

# 2. Creación de la columna booleana 'en_revision'
suelo['en_revision'] = cond_ph_ausente | cond_ph_fuera_rango | cond_fecha_invalida

# 3. Conteo por motivo individual
n_ph_ausente = cond_ph_ausente.sum()
n_ph_fuera_rango = cond_ph_fuera_rango.sum()
n_fecha_invalida = cond_fecha_invalida.sum()

print("--- REPORTE DE MOTIVOS INDIVIDUALES DE REVISIÓN ---")
print(f"1. Filas con pH ausente: {n_ph_ausente:,}")
print(f"2. Filas con pH fuera de rango [0, 14]: {n_ph_fuera_rango:,}")
print(f"3. Filas con Fecha no interpretable: {n_fecha_invalida:,}")
print(f"Suma acumulada de motivos (puede haber solapamientos): {n_ph_ausente + n_ph_fuera_rango + n_fecha_invalida:,}")

# 4. Conteo general y comprobación del universo original
conteos = suelo['en_revision'].value_counts()
marcadas = conteos.get(True, 0)
no_marcadas = conteos.get(False, 0)
total_filas = len(suelo)

print("\n--- RESUMEN GENERAL DE 'en_revision' ---")
print(f"Total filas marcadas (en_revision = True): {marcadas:,} ({(marcadas/total_filas)*100:.2f}%)")
print(f"Total filas válidas (en_revision = False): {no_marcadas:,} ({(no_marcadas/total_filas)*100:.2f}%)")
print(f"Total filas evaluadas: {total_filas:,}")

# Comprobación estricta
assert marcadas + no_marcadas == total_filas, "Error: La suma de marcadas y no marcadas no coincide con el total original."
print("\n✅ COMPROBACIÓN EXITOSA: La suma de marcadas y no marcadas es idéntica al total original de 92.738 filas.")

--- REPORTE DE MOTIVOS INDIVIDUALES DE REVISIÓN ---
1. Filas con pH ausente: 11
2. Filas con pH fuera de rango [0, 14]: 0
3. Filas con Fecha no interpretable: 0
Suma acumulada de motivos (puede haber solapamientos): 11

--- RESUMEN GENERAL DE 'en_revision' ---
Total filas marcadas (en_revision = True): 11 (0.01%)
Total filas válidas (en_revision = False): 92,727 (99.99%)
Total filas evaluadas: 92,738

✅ COMPROBACIÓN EXITOSA: La suma de marcadas y no marcadas es idéntica al total original de 92.738 filas.


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

## 5. Evidencias y conclusiones · 10 minutos

Construir `controles` con las columnas `control` y `cantidad`: total de entrada, pH sin número, pH fuera del intervalo, fecha no interpretable y total de filas en revisión. Exportar a `OUT / "controles_agrosavia.csv"` sin índice. Redactar dos hallazgos apoyados en los conteos.

📘 **Apoyo:** `pd.DataFrame`, `to_csv(index=False)` y las tablas de las actividades anteriores.

✅ **Comprobación:** El archivo se genera dentro de la ruta `OUT`. Cada hallazgo menciona un conteo y explica su significado o limitación.

In [9]:
import pandas as pd

# 1. Construcción de la tabla de controles con los 5 indicadores exactos
datos_controles = {
    'control': [
        'total de entrada',
        'pH sin número',
        'pH fuera del intervalo',
        'fecha no interpretable',
        'total de filas en revisión'
    ],
    'cantidad': [
        len(suelo),
        int(cond_ph_ausente.sum()),
        int(cond_ph_fuera_rango.sum()),
        int(cond_fecha_invalida.sum()),
        int(suelo['en_revision'].sum())
    ]
}

controles = pd.DataFrame(datos_controles)

# 2. Exportación a la ruta OUT definida al inicio del notebook
ruta_archivo_out = OUT / "controles_agrosavia.csv"
controles.to_csv(ruta_archivo_out, index=False)

print(f"--- TABLA DE CONTROLES GENERADA ---")
display(controles)
print(f"\n✅ Archivo exportado exitosamente en: {ruta_archivo_out}")

--- TABLA DE CONTROLES GENERADA ---


,control,cantidad
0,total de entrada,92738
1,pH sin número,11
2,pH fuera del intervalo,0
3,fecha no interpretable,0
4,total de filas en revisión,11



✅ Archivo exportado exitosamente en: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD01/controles_agrosavia.csv


**Interpretación del resultado:**

_Pendiente de completar con evidencia de las tablas o controles anteriores._

## Entrega

El notebook completado se conserva como `soluciones/PD01_exploracion_agrosavia.ipynb`. La salida regenerable es `kit/salidas/soluciones/PD01/controles_agrosavia.csv`.

✅ Deben quedar visibles el perfil, los controles de conversión, la conciliación de filas y dos hallazgos. El notebook debe ejecutar todas las celdas en orden después de reiniciar el kernel.

⚠️ El intervalo de pH es un control de dominio para esta práctica. No determina por sí solo una recomendación agronómica. La frecuencia de análisis de suelo no representa automáticamente la distribución de los suelos de todo el territorio.